In [2]:
## 数据导入
import pandas as pd
import numpy as np

train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train.csv')  # 读取csv文件

In [3]:
train.info()  # 查看数据基本信息

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            127744 non-null  int64  
 1   target           127744 non-null  int64  
 2   timestamp        127744 non-null  float64
 3   processId        127744 non-null  int64  
 4   threadId         127744 non-null  int64  
 5   parentProcessId  127744 non-null  int64  
 6   userId           127744 non-null  int64  
 7   mountNamespace   127744 non-null  int64  
 8   processName      127744 non-null  object 
 9   hostName         127744 non-null  int64  
 10  eventId          127744 non-null  int64  
 11  eventName        127744 non-null  object 
 12  stackAddresses   127744 non-null  object 
 13  argsNum          127744 non-null  int64  
 14  returnValue      127744 non-null  int64  
 15  args             127744 non-null  object 
dtypes: float64(1), int64(11), object(4)
me

In [4]:
## 特征工程（涉及 缺失数据， 数据编码， 异常数据处理etc）

In [5]:
### missing data imputation 
### using feature-engine
from sklearn.impute import SimpleImputer

# 查看缺失值
train.isnull().sum()

index              0
target             0
timestamp          0
processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
processName        0
hostName           0
eventId            0
eventName          0
stackAddresses     0
argsNum            0
returnValue        0
args               0
dtype: int64

In [6]:
train['args'][0] ## Python 的 list of dicts（字符串形式）

"[{'name': 'domain', 'type': 'int', 'value': 'AF_UNIX'}, {'name': 'type', 'type': 'int', 'value': 'SOCK_DGRAM|SOCK_CLOEXEC'}, {'name': 'protocol', 'type': 'int', 'value': 0}]"

In [7]:
import ast
import pandas as pd

# 假设 train 已经有 args 列
train['args_parsed'] = train['args'].apply(ast.literal_eval)


In [8]:
# 转换成 {name: value} 的字典
def list_to_dict(lst):
    return {d['name']: d['value'] for d in lst}

train['args_dict'] = train['args_parsed'].apply(list_to_dict)

# 展开成 DataFrame
args_df = pd.json_normalize(train['args_dict'])


In [9]:
args_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   domain      451 non-null    object 
 1   type        451 non-null    object 
 2   protocol    451 non-null    float64
 3   pathname    65560 non-null  object 
 4   flags       51280 non-null  object 
 5   dev         14324 non-null  float64
 6   inode       14324 non-null  float64
 7   fd          57149 non-null  float64
 8   statbuf     26285 non-null  object 
 9   dirfd       36772 non-null  float64
 10  mode        43398 non-null  object 
 11  ruid        4 non-null      float64
 12  euid        4 non-null      float64
 13  rgid        11 non-null     float64
 14  egid        11 non-null     float64
 15  cap         2832 non-null   object 
 16  sockfd      591 non-null    float64
 17  addr        591 non-null    object 
 18  addrlen     591 non-null    object 
 19  dirp        785 non-nul

In [10]:
train = pd.concat([train.drop(columns=['args', 'args_parsed', 'args_dict']), args_df], axis=1)


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 53 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            127744 non-null  int64  
 1   target           127744 non-null  int64  
 2   timestamp        127744 non-null  float64
 3   processId        127744 non-null  int64  
 4   threadId         127744 non-null  int64  
 5   parentProcessId  127744 non-null  int64  
 6   userId           127744 non-null  int64  
 7   mountNamespace   127744 non-null  int64  
 8   processName      127744 non-null  object 
 9   hostName         127744 non-null  int64  
 10  eventId          127744 non-null  int64  
 11  eventName        127744 non-null  object 
 12  stackAddresses   127744 non-null  object 
 13  argsNum          127744 non-null  int64  
 14  returnValue      127744 non-null  int64  
 15  domain           451 non-null     object 
 16  type             451 non-null     obje

In [12]:
train.isnull().sum()

index                   0
target                  0
timestamp               0
processId               0
threadId                0
parentProcessId         0
userId                  0
mountNamespace          0
processName             0
hostName                0
eventId                 0
eventName               0
stackAddresses          0
argsNum                 0
returnValue             0
domain             127293
type               127293
protocol           127293
pathname            62184
flags               76464
dev                113420
inode              113420
fd                  70595
statbuf            101459
dirfd               90972
mode                84346
ruid               127740
euid               127740
rgid               127733
egid               127733
cap                124912
sockfd             127153
addr               127153
addrlen            127153
dirp               126959
count              126959
stack              127552
parent_tid         127552
child_tid   

In [13]:
train.size

6770432

In [14]:
# 5. 导出成新的 CSV
train.to_csv("D:/NUSMaster/semester1/CS5344BigDataAnalytic/Projects/sample_processes_train_parsed.csv", index=False)

print("✅ 已经生成新的 CSV 文件: sample_processes_train_parsed.csv")

✅ 已经生成新的 CSV 文件: sample_processes_train_parsed.csv


In [15]:
# 查看当前train中的数值型特征是否有缺失值
# 只选择数值型列
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
# 统计每个数值型列的缺失值个数
missing_numeric = train[num_cols].isnull().sum()
# 只显示有缺失的列
missing_numeric = missing_numeric[missing_numeric > 0]
print(missing_numeric)

target      127743
protocol    127293
dev         113420
inode       113420
fd           70595
dirfd        90972
ruid        127740
euid        127740
rgid        127733
egid        127733
sockfd      127153
count       126959
tls         127552
arg2        127333
arg3        127333
arg4        127333
arg5        127333
pid         127552
sig         127552
oldfd       127601
newfd       127612
uid         127737
gid         127741
dtype: int64


In [43]:
import numpy as np
import pandas as pd
## 读取 parsed csv
train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train_parsed.csv')  # 读取csv文件
## 查看非数值型特征都有哪些列，每一列都有什么
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    unique_values = train[col].unique()
    #print(f"Column: {col}, Unique Values: {unique_values}\n")
    

C:\Users\fkbgr\AppData\Local\Temp\ipykernel_30408\3434952398.py:4: DtypeWarning: Columns (47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train_parsed.csv')  # 读取csv文件


In [44]:
#############################################################################
### 异常值处理 - 缺失值>95%删除
#############################################################################
#############################################################################
import pandas as pd

# 假设 df 是你的 DataFrame
# 删除缺失比例超过 95% 的列
print("筛选前剩余的列数:", train.shape[1])
# 查看都有哪些列
print("删除前都有哪些列:", train.columns)
threshold = 0.95
train = train.loc[:, train.isnull().mean() <= threshold]
print("筛选后剩余的列数:", train.shape[1])
print("删除后都有哪些列:", train.columns)


筛选前剩余的列数: 53
删除前都有哪些列: Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum',
       'returnValue', 'domain', 'type', 'protocol', 'pathname', 'flags', 'dev',
       'inode', 'fd', 'statbuf', 'dirfd', 'mode', 'ruid', 'euid', 'rgid',
       'egid', 'cap', 'sockfd', 'addr', 'addrlen', 'dirp', 'count', 'stack',
       'parent_tid', 'child_tid', 'tls', 'option', 'arg2', 'arg3', 'arg4',
       'arg5', 'pid', 'sig', 'target.1', 'oldfd', 'newfd', 'uid', 'argv',
       'gid'],
      dtype='object')
筛选后剩余的列数: 23
删除后都有哪些列: Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum',
       'returnValue', 'pathname', 'flags', 'dev', 'inode', 'fd', 'statbuf',
       'dirfd', 'mode'],
      dtype='object')


In [45]:
'''
当前文本特征包括：processName, eventName, stackAddress, pathname, flags, statbuf, mode
'''
# 查看当前train中的文本型特征
cat_cols = train.select_dtypes(include=['object']).columns
print("当前文本特征包括：", cat_cols)

当前文本特征包括： Index(['processName', 'eventName', 'stackAddresses', 'pathname', 'flags',
       'statbuf', 'mode'],
      dtype='object')


In [46]:
#############################################################################
### 文本特征处理 
#############################################################################
#############################################################################

In [47]:
import numpy as np
import pandas as pd
## 读取 parsed csv
#train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train_parsed.csv')  # 读取csv文件
########################################################
### processName、eventName
########################################################
# 使用 onehotencoder 对 processName和 eventName 进行编码
from sklearn.preprocessing import LabelEncoder
for col in ["processName", "eventName"]:
    train[col] = train[col].astype(str)
    le = LabelEncoder()
    train[col + "_enc"] = le.fit_transform(train[col])


In [48]:
# 查看编码后的数据列 processName_ 和 eventName_
print(train.columns)

Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum',
       'returnValue', 'pathname', 'flags', 'dev', 'inode', 'fd', 'statbuf',
       'dirfd', 'mode', 'processName_enc', 'eventName_enc'],
      dtype='object')


In [49]:
# 查看第一条数据
print(train[["processName", "processName_enc", "eventName", "eventName_enc"]].head(1))

       processName  processName_enc eventName  eventName_enc
0  systemd-resolve               21    socket             28


In [50]:
########################################################
### domain、type、cap、option
########################################################


In [51]:
# 查看前几行数据
print(train.head())

   index  target   timestamp  processId  threadId  parentProcessId  userId  \
0      0       0  124.439221        381       381                1     101   
1      2       0  124.439958          1         1                0       0   
2      4       0  124.440037          1         1                0       0   
3      6       0  124.440379          1         1                0       0   
4      7       0  124.440414          1         1                0       0   

   mountNamespace      processName  hostName  ...          pathname  \
0      4026532232  systemd-resolve         0  ...               NaN   
1      4026531840          systemd         0  ...  /proc/378/cgroup   
2      4026531840          systemd         0  ...               NaN   
3      4026531840          systemd         0  ...  /proc/381/cgroup   
4      4026531840          systemd         0  ...  /proc/381/cgroup   

                  flags  dev    inode    fd         statbuf  dirfd  \
0                   NaN  NaN      

In [52]:
########################################################
### flags
########################################################
# flags 列是以 | 分隔的多值类别特征，如 'O_RDONLY|O_NONBLOCK|O_DIRECTORY'，使用 MultiLabelBinarizer 进行编码，需要拆分 '|' 并转为多热（multi-hot）向量
# from sklearn.preprocessing import MultiLabelBinarizer
# train["flags_split"] = train["flags"].dropna().apply(lambda x: x.split("|"))
# mlb = MultiLabelBinarizer()
# flags_encoded = mlb.fit_transform(train["flags_split"])
# flags_df = pd.DataFrame(flags_encoded, columns=[f"flag_{f}" for f in mlb.classes_])
# train = pd.concat([train, flags_df], axis=1)

In [ ]:
########################################################
### addrlen, dirp, stack, parent_tid, child_tid, statbuf等直接删除
########################################################
#train = train.drop(columns=["statbuf"])
print('查看删除statbuf后剩余的列数:', train.shape[1])

查看删除statbuf后剩余的列数: 25


In [55]:
# 查看当前特征中都有哪些列
print(train.columns)
# 查看当前特征中都是哪些类型
print(train.dtypes)

Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum',
       'returnValue', 'pathname', 'flags', 'dev', 'inode', 'fd', 'statbuf',
       'dirfd', 'mode', 'processName_enc', 'eventName_enc'],
      dtype='object')
index                int64
target               int64
timestamp          float64
processId            int64
threadId             int64
parentProcessId      int64
userId               int64
mountNamespace       int64
processName         object
hostName             int64
eventId              int64
eventName           object
stackAddresses      object
argsNum              int64
returnValue          int64
pathname            object
flags               object
dev                float64
inode              float64
fd                 float64
statbuf             object
dirfd              float64
mode                object
processName_enc

In [56]:
########################################################
### stackAddress列表，转成栈深度
########################################################
# stackAddress 列是一个栈地址列表，使用栈深度（列表长度）作为数值特征
import ast
import pandas as pd

def extract_stack_depth(x):
    """
    将 stackAddresses 解析成列表，并返回其深度（长度）。
    - 空值或格式错误的返回 0
    """
    if isinstance(x, str) and x.startswith('['):
        try:
            vals = ast.literal_eval(x)
            if isinstance(vals, list):
                return len(vals)
        except:
            pass
    return 0

train["stack_depth"] = train["stackAddresses"].apply(extract_stack_depth)


In [59]:
# 查看当前特征中都有哪些列
print(train.columns)
# 查看当前特征中都是哪些类型
print(train.dtypes)
# 查看前几行数据
print(train.head())

Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName', 'eventId', 'eventName', 'stackAddresses', 'argsNum',
       'returnValue', 'pathname', 'flags', 'dev', 'inode', 'fd', 'statbuf',
       'dirfd', 'mode', 'processName_enc', 'eventName_enc', 'stack_depth'],
      dtype='object')
index                int64
target               int64
timestamp          float64
processId            int64
threadId             int64
parentProcessId      int64
userId               int64
mountNamespace       int64
processName         object
hostName             int64
eventId              int64
eventName           object
stackAddresses      object
argsNum              int64
returnValue          int64
pathname            object
flags               object
dev                float64
inode              float64
fd                 float64
statbuf             object
dirfd              float64
mode                object


In [61]:
########################################################
### 进行 flags 列的解码
########################################################
import pandas as pd

def decode_or_split_flags(value):
    """
    同时支持字符串型和整型 flags:
    - 对字符串（'O_RDONLY|O_CLOEXEC'）自动拆分
    - 对整数（如 -1868620656）按 Linux open flags 解析
    - 对 NaN / 空值返回 []
    """

    # ========== 1) 空值直接返回 ==========
    if pd.isna(value):
        return []

    # ========== 2) 如果是字符串 ==========
    if isinstance(value, str):
        # 拆分多标志
        if "|" in value:
            return [v.strip() for v in value.split("|") if v.strip()]
        # 单个标志（如 "O_RDONLY"）
        elif value.strip():
            return [value.strip()]
        else:
            return []

    # ========== 3) 如果是数值型（含负数） ==========
    try:
        value = int(value) & 0xFFFFFFFF  # 转无符号 32 位
    except Exception:
        return []

    FLAG_MAP = {
        0x00000000: "O_RDONLY",
        0x00000001: "O_WRONLY",
        0x00000002: "O_RDWR",
        0x00000040: "O_CREAT",
        0x00000080: "O_EXCL",
        0x00000100: "O_NOCTTY",
        0x00000200: "O_TRUNC",
        0x00000400: "O_APPEND",
        0x00000800: "O_NONBLOCK",
        0x00001000: "O_DSYNC",
        0x00002000: "FASYNC",
        0x00004000: "O_DIRECT",
        0x00008000: "O_LARGEFILE",
        0x00010000: "O_DIRECTORY",
        0x00020000: "O_NOFOLLOW",
        0x00040000: "O_NOATIME",
        0x00080000: "O_CLOEXEC",
        0x00100000: "O_PATH",
        0x00200000: "O_TMPFILE"
    }

    result = [name for mask, name in FLAG_MAP.items() if value & mask]

    # 没有匹配项默认 O_RDONLY
    if not result:
        result.append("O_RDONLY")

    return result


train["decoded_flags"] = train["flags"].apply(decode_or_split_flags)
# 查看前几行数据
print(train[["flags", "decoded_flags"]].head())


                  flags            decoded_flags
0                   NaN                       []
1  O_RDONLY|O_LARGEFILE  [O_RDONLY, O_LARGEFILE]
2                   NaN                       []
3  O_RDONLY|O_LARGEFILE  [O_RDONLY, O_LARGEFILE]
4    O_RDONLY|O_CLOEXEC    [O_RDONLY, O_CLOEXEC]


In [65]:

########################################################
### 进行 flags 列进行多列二进制编码
########################################################
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
flags_encoded = mlb.fit_transform(train["decoded_flags"])
flags_df = pd.DataFrame(flags_encoded, columns=[f"flag_{c}" for c in mlb.classes_])

train = pd.concat([train, flags_df], axis=1)
# 查看当前特征中都有哪些列
print(train.columns)
# 查看当前前几行数据
print(train.head())


Index(['index', 'target', 'timestamp', 'processId', 'threadId',
       'parentProcessId', 'userId', 'mountNamespace', 'processName',
       'hostName',
       ...
       'flag_O_LARGEFILE', 'flag_O_NOATIME', 'flag_O_NOCTTY',
       'flag_O_NOFOLLOW', 'flag_O_NONBLOCK', 'flag_O_PATH', 'flag_O_RDONLY',
       'flag_O_RDWR', 'flag_O_TRUNC', 'flag_O_WRONLY'],
      dtype='object', length=175)
   index  target   timestamp  processId  threadId  parentProcessId  userId  \
0      0       0  124.439221        381       381                1     101   
1      2       0  124.439958          1         1                0       0   
2      4       0  124.440037          1         1                0       0   
3      6       0  124.440379          1         1                0       0   
4      7       0  124.440414          1         1                0       0   

   mountNamespace      processName  hostName  ...  flag_O_LARGEFILE  \
0      4026532232  systemd-resolve         0  ...                 0 

In [66]:
####################################################
### 暂时放弃 pathname 和 mode 数据列的预处理
### 保存最终处理后的数据
####################################################
### 删除pathname和mode列
train = train.drop(columns=["pathname", "mode"])


#train.to_csv("processed_train_data.csv", index=False)

In [69]:
# 查看所有列, 一条一条显示
for col in train.columns.tolist():
    print(col)

index
target
timestamp
processId
threadId
parentProcessId
userId
mountNamespace
processName
hostName
eventId
eventName
stackAddresses
argsNum
returnValue
flags
dev
inode
fd
statbuf
dirfd
processName_enc
eventName_enc
stack_depth
decoded_flags
flag_-1107596144
flag_-1369736048
flag_-1868620656
flag_-234652528
flag_-385483702
flag_-958931824
flag_0
flag_1222958224
flag_1462094922
flag_2
flag_512
flag_CLONE_CHILD_CLEARTID
flag_CLONE_CHILD_SETTID
flag_CLONE_FILES
flag_CLONE_FS
flag_CLONE_PARENT_SETTID
flag_CLONE_SETTLS
flag_CLONE_SIGHAND
flag_CLONE_SYSVSEM
flag_CLONE_THREAD
flag_CLONE_VFORK
flag_CLONE_VM
flag_O_APPEND
flag_O_CLOEXEC
flag_O_CREAT
flag_O_DIRECTORY
flag_O_EXCL
flag_O_LARGEFILE
flag_O_NOATIME
flag_O_NOCTTY
flag_O_NOFOLLOW
flag_O_NONBLOCK
flag_O_PATH
flag_O_RDONLY
flag_O_RDWR
flag_O_TRUNC
flag_O_WRONLY
flag_-1107596144
flag_-1369736048
flag_-1868620656
flag_-234652528
flag_-385483702
flag_-958931824
flag_0
flag_1222958224
flag_1462094922
flag_2
flag_512
flag_CLONE_CHILD_CLEARTI

In [71]:
# 查看还有哪些列是文本型
cat_cols = train.select_dtypes(include=['object']).columns
print("当前文本特征包括：", cat_cols)
# 删除这些列
train = train.drop(columns=cat_cols)

当前文本特征包括： Index(['processName', 'eventName', 'stackAddresses', 'flags', 'statbuf',
       'decoded_flags'],
      dtype='object')


In [72]:
# 查看所有列, 一条一条显示
for col in train.columns.tolist():
    print(col)

index
target
timestamp
processId
threadId
parentProcessId
userId
mountNamespace
hostName
eventId
argsNum
returnValue
dev
inode
fd
dirfd
processName_enc
eventName_enc
stack_depth
flag_-1107596144
flag_-1369736048
flag_-1868620656
flag_-234652528
flag_-385483702
flag_-958931824
flag_0
flag_1222958224
flag_1462094922
flag_2
flag_512
flag_CLONE_CHILD_CLEARTID
flag_CLONE_CHILD_SETTID
flag_CLONE_FILES
flag_CLONE_FS
flag_CLONE_PARENT_SETTID
flag_CLONE_SETTLS
flag_CLONE_SIGHAND
flag_CLONE_SYSVSEM
flag_CLONE_THREAD
flag_CLONE_VFORK
flag_CLONE_VM
flag_O_APPEND
flag_O_CLOEXEC
flag_O_CREAT
flag_O_DIRECTORY
flag_O_EXCL
flag_O_LARGEFILE
flag_O_NOATIME
flag_O_NOCTTY
flag_O_NOFOLLOW
flag_O_NONBLOCK
flag_O_PATH
flag_O_RDONLY
flag_O_RDWR
flag_O_TRUNC
flag_O_WRONLY
flag_-1107596144
flag_-1369736048
flag_-1868620656
flag_-234652528
flag_-385483702
flag_-958931824
flag_0
flag_1222958224
flag_1462094922
flag_2
flag_512
flag_CLONE_CHILD_CLEARTID
flag_CLONE_CHILD_SETTID
flag_CLONE_FILES
flag_CLONE_FS
flag_CLO

In [73]:
# 对数据进行保存
train.to_csv("processed_train_data_all2Number_delProcessname_delMode.csv", index=False)
print("✅ 已经生成最终处理后的 CSV 文件: processed_train_data_all2Number_delProcessname_delMode.csv")

✅ 已经生成最终处理后的 CSV 文件: processed_train_data_all2Number_delProcessname_delMode.csv
